# CS781 Capstone · Phase 5
## Prefix teacher for legal statute identification

This notebook exports the predictions of the trained **prefix LoRMA+ model** for knowledge distillation into the efficient student. The teacher produces one logit for each of the 100 statute labels, allowing several statutes to apply to the same case.

I use the selected EMA checkpoint from `lsi_prefix_lorma_001` as the teacher. It provides a larger, bidirectional document model while the student uses a compact backbone derived from the efficient Phase 1 model. The export keeps the teacher's learned parameters fixed and runs over the training split.

The output is `lsi_prefix_lorma_001_teacher_cache.zip`, containing the training logits and the information needed to align them with the student's cases and statute columns. Student training is carried out separately.


### Kaggle inputs

Set `PREPARED_INPUT` to the bundle from [preparation](phase5_preparation.ipynb) and `TEACHER_BACKUP_INPUT` to the selected backup from [prefix training](phase5_prefix_lorma.ipynb). The archived run used the selected checkpoint at update 2,000. The exported teacher cache goes into the [efficient KD notebook](phase5_efficient_kd.ipynb).


### Teacher configuration

The teacher starts from the original Phase 1 prefix language model. Its Phase 5 adaptation uses output-side multiplicative updates,

\[
W_{\mathrm{eff}} = \left(I + \frac{\alpha}{r}BA\right)W_0,
\]

with rank \(r=8\), \(\alpha=8\), and adaptation dropout 0.05. The original backbone weights remain frozen during task adaptation. The document head combines contextualized chunk representations with label-wise attention and produces independent statute logits.

| Setting | Value |
|---|---|
| Tokenizer | Original legal BPE, 16,000 tokens |
| Document coverage | Up to four uniformly selected chunks |
| Chunk length / overlap | 512 positions / 64 tokens |
| Chunk encoding | Batches shared across cases; up to 64 chunks per GPU call |
| Export case batch | 32 across T4 ×2 |
| Selected teacher weights | EMA, update 2000 |

Threshold calibration is used for the teacher's hard-label evaluation. The distillation cache stores **raw logits**, before applying those thresholds.


## 1. Inputs and project setup

The export requires the prepared Phase 5 data, the original Phase 1 prefix checkpoint, and the completed Phase 5 prefix task backup. A task backup stores the learned adaptation and classification tensors; the original Phase 1 checkpoint supplies the frozen backbone.

The paths below correspond to the attached Kaggle datasets. `CHECKPOINT_INPUT = None` enables discovery of the original prefix checkpoint. A dedicated working directory keeps this export separate from other experiments.


In [1]:
from pathlib import Path

RUN_ID = "lsi_prefix_lorma_001"
PROJECT_ROOT = Path("/kaggle/working/cs781_phase5_prefix_teacher")
PROJECT_CODE_ROOT = Path("/kaggle/working/cs781_phase5_project")
INPUT_ROOT = Path("/kaggle/input")

PREPARED_INPUT = "/kaggle/input/datasets/ash9201k/phase5-dataset/phase5_lsi_prepared"
TEACHER_BACKUP_INPUT = "/kaggle/input/datasets/ash9201k/phase5-prefix/lsi_prefix_lorma_001"
CHECKPOINT_INPUT = None

EXPECTED_TEACHER_SHA256 = "4ca566e8fb2f6ea1c81eaae39ff6453674ed0bd71bde79b931695369d7c43645"


### Portable project source

The collapsed cell contains the project source snapshot used for this experiment. It writes the shared model, data and evaluation modules to `PROJECT_CODE_ROOT`. Its archive checksum is checked before extraction.


In [2]:
import base64
import hashlib
import io
import zipfile

SOURCE_ARCHIVE_B64 = "..."
source_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
assert hashlib.sha256(source_bytes).hexdigest() == "c87cca770e9cfeb6067410eb14c7de661c98204a1551096bbd360a46f4c99b57"
PROJECT_CODE_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    for member in archive.infolist():
        destination = (PROJECT_CODE_ROOT / member.filename).resolve()
        if not destination.is_relative_to(PROJECT_CODE_ROOT.resolve()):
            raise ValueError("Unexpected project archive path")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))

print("Project source", PROJECT_CODE_ROOT / "src")


Project source /kaggle/working/cs781_phase5_project/src


## 2. Prepare the teacher and run configuration

The input files are staged into an isolated directory and checked before use. The training-case list is constructed deterministically from the packed index and checked against the fingerprint saved with the teacher. Statute order, loss settings and the selected checkpoint must also match the completed run.

The export inherits the saved teacher configuration, changing only input/output paths and the action to `export_teacher`. This preserves the original architecture and classification head. The model is evaluated with fixed weights; there are no optimizer updates in this notebook.


In [3]:
import contextlib
import io
import json
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import time

import pandas as pd
import torch
from IPython.display import display

sys.path.insert(0, str(PROJECT_CODE_ROOT / "src"))
from common import read_json, write_json, sha256
from lsi_data import digest, training_selection
from lsi_experiment import restore_run, protocol
from lsi_prepare import restore_prepared


def run_command(arguments, compact=False):
    process = subprocess.Popen(
        arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    worker_log = []

    def show(line):
        if not compact:
            return True
        progress = re.match(r"Teacher cache committed ([\d,]+)/([\d,]+)", line)
        if progress:
            completed, total = [int(value.replace(",", "")) for value in progress.groups()]
            return completed == 1024 or completed % 10240 == 0 or completed == total
        return line.startswith((
            "Verifying original Phase 1 checkpoint", "Run ",
            "Teacher-cache ZIP", "Task backup", "Diagnostics",
        ))

    try:
        for line in process.stdout:
            worker_log.append(line)
            if show(line):
                print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        for line in process.stdout:
            print(line, end="", flush=True)
        process.wait()
        raise
    if returncode != 0:
        if compact:
            print("".join(worker_log[-60:]), end="")
        raise RuntimeError("The export worker did not complete; inspect the run's runtime.log.")


PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(
    prefix="prefix_teacher_inputs_", dir=str(PROJECT_ROOT.parent)
) as temporary:
    stage = Path(temporary)
    fresh_run = stage / "runs" / RUN_ID
    with contextlib.redirect_stdout(io.StringIO()):
        restore_run({
            "project_root": str(stage), "resume_input": TEACHER_BACKUP_INPUT,
        }, fresh_run)
        prepared = restore_prepared({
            "project_root": str(stage), "input_root": str(INPUT_ROOT),
            "prepared_input": PREPARED_INPUT,
        })
    if not prepared:
        raise FileNotFoundError("Set PREPARED_INPUT to the prepared Phase 5 ZIP or folder.")

    teacher_config = read_json(fresh_run / "config.json")
    reference = teacher_config["protocol"]["inputs"]
    if teacher_config["run_id"] != RUN_ID:
        raise ValueError("The task backup does not match the selected teacher run.")
    if sha256(fresh_run / "best.pt") != EXPECTED_TEACHER_SHA256:
        raise ValueError("The selected teacher checkpoint checksum differs.")
    for name in ("best.pt", "last.pt"):
        state = torch.load(fresh_run / name, map_location="cpu", weights_only=False)
        if state["protocol_hash"] != teacher_config["protocol_hash"]:
            raise ValueError(f"{name} has a different experiment identity.")
        del state

    cache = stage / "prepared" / "cache"
    if read_json(cache / "manifest.json")["fingerprint"] != reference["cache_fingerprint"]:
        raise ValueError("The prepared cache differs from the teacher's data.")
    rows = read_json(cache / "index.json")["cases"]
    row_uids = [row["row_uid"] for row in training_selection(rows, True)]
    fingerprint = digest(row_uids)
    if fingerprint != reference["training_selection_fingerprint"]:
        raise ValueError("Training-case identities or their order differ.")
    write_json(stage / "reports" / "training_selection.json", {
        "row_uids": row_uids, "fingerprint": fingerprint,
    })
    if read_json(stage / "reports" / "label_vocabulary.json")["label_ids"] != reference["label_ids"]:
        raise ValueError("The statute columns differ from the teacher's columns.")
    if sha256(stage / "reports" / "loss_settings.json") != reference["loss_settings_sha256"]:
        raise ValueError("The saved training-label statistics differ.")

    previous = PROJECT_ROOT.parent / f"prefix_teacher_previous_{time.time_ns()}"
    for relative in (Path("prepared/cache"), Path("reports"), Path("metadata"), Path("runs") / RUN_ID):
        target = PROJECT_ROOT / relative
        if target.exists():
            saved_target = previous / relative
            saved_target.parent.mkdir(parents=True, exist_ok=True)
            target.rename(saved_target)
        target.parent.mkdir(parents=True, exist_ok=True)
        (stage / relative).rename(target)

run_dir = PROJECT_ROOT / "runs" / RUN_ID
settings = teacher_config["requested_settings"].copy()
settings.update(
    project_root=str(PROJECT_ROOT), input_root=str(INPUT_ROOT),
    prepared_input=PREPARED_INPUT, checkpoint_input=CHECKPOINT_INPUT,
    action="export_teacher", resume_input=None, export_teacher_cache=False,
)
request_path = PROJECT_CODE_ROOT / "configs" / f"{RUN_ID}_teacher_request.json"
write_json(request_path, settings)


### Selected teacher checkpoint

These values are read from the completed prefix run's saved configuration, training status and development metrics. They describe the teacher checkpoint used for export; development evaluation is not repeated here.


In [4]:
teacher_status = read_json(run_dir / "training_status.json")
teacher_metrics = read_json(run_dir / "exports" / "metrics.json")
teacher_identity = read_json(run_dir / "exports" / "export_identity.json")

teacher_summary = pd.DataFrame({
    "Item": [
        "Teacher run", "Selected update", "Evaluation weights",
        "Total parameters", "Trainable parameters",
        "Dev macro-F1 at 0.5", "Dev macro-F1 with calibrated thresholds",
    ],
    "Value": [
        teacher_config["run_id"], str(teacher_status["selected_update"]),
        teacher_identity["evaluation_weights"],
        f"{teacher_config['parameters']['total_parameters']:,}",
        f"{teacher_config['parameters']['trainable_parameters']:,}",
        f"{teacher_metrics['fixed_05']['macro_f1']:.6f}",
        f"{teacher_metrics['regularized_per_label']['macro_f1']:.6f}",
    ],
})
display(teacher_summary.set_index("Item"))


,Value
Item,
Teacher run,lsi_prefix_lorma_001
Selected update,2000
Evaluation weights,EMA
Total parameters,"221,091,492"
Trainable parameters,"3,229,348"
Dev macro-F1 at 0.5,0.216408
Dev macro-F1 with calibrated thresholds,0.247537


## 3. Verify the prepared inputs

The worker verifies the packed token files, training-case fingerprint, statute columns and tokenizer signature. It also checks the original Phase 1 prefix checkpoint against its recorded SHA-256 checksum.


In [5]:
input_report = PROJECT_CODE_ROOT / "current_inputs.json"
run_command([
    sys.executable, "-u", str(PROJECT_CODE_ROOT / "src/lsi_training.py"),
    "--inspect", str(request_path), "--report", str(input_report),
])
inputs_record = read_json(input_report)
display(pd.DataFrame([{
    "train_cases": inputs_record["train_cases"],
    "dev_cases": inputs_record["dev_cases"],
    "test_cases": inputs_record["test_cases"],
    "statute_labels": len(inputs_record["label_ids"]),
    "tokenizer_vocabulary": inputs_record["tokenizer_signature"]["vocab_size"],
}]))


Verifying original Phase 1 checkpoint prefix
Prepared cases 42750 10181 13019
Statute labels 100


,train_cases,dev_cases,test_cases,statute_labels,tokenizer_vocabulary
0,42750,10181,13019,100,16000


## 4. Export the training logits

For each training case \(i\), the selected teacher produces a vector \(z_{T,i}\in\mathbb{R}^{100}\). The worker loads the EMA parameters, switches the model to evaluation mode and runs without gradient updates. It uses the same chunk geometry as the completed teacher run.

The export records the training-row identifiers, ordered statute IDs, tokenizer signature, teacher checkpoint checksum and data fingerprints. These accompany the logits so that the student can verify alignment before using them. The output below shows selected progress messages from the recorded export; the worker also keeps its full runtime log.


In [6]:
settings["checkpoint_input"] = inputs_record["checkpoint_path"]
if digest(protocol(settings, {"record": inputs_record})) != teacher_config["protocol_hash"]:
    raise ValueError("The current inputs or project source differ from the saved teacher experiment.")
write_json(request_path, settings)

run_command([
    sys.executable, "-u", str(PROJECT_CODE_ROOT / "src/lsi_training.py"),
    "--config", str(request_path),
], compact=True)


Verifying original Phase 1 checkpoint prefix
Run lsi_prefix_lorma_001
Teacher cache committed 1,024/42,750
Teacher cache committed 10,240/42,750
Teacher cache committed 20,480/42,750
Teacher cache committed 30,720/42,750
Teacher cache committed 40,960/42,750
Teacher cache committed 42,750/42,750
Teacher-cache ZIP /kaggle/working/lsi_prefix_lorma_001_teacher_cache.zip
Task backup /kaggle/working/lsi_prefix_lorma_001_backup.zip
Diagnostics /kaggle/working/lsi_prefix_lorma_001_diagnostics.zip


## 5. Cache output and student handoff

The export completed for **42,750 training cases**. Each case has one teacher logit for each of the **100 statute labels**. The cache is written to:

```text
/kaggle/working/lsi_prefix_lorma_001_teacher_cache.zip
```

| File in the ZIP | Purpose |
|---|---|
| `teacher_logits_train.npz` | Training logits, row identifiers and ordered statute IDs |
| `teacher_identity.json` | Teacher identity, data/tokenizer fingerprints, geometry and logit-file checksum |

The distillation targets are formed from the raw teacher logits, separately for each statute:

\[
q_{ic}=\sigma\!\left(\frac{z_{T,ic}}{T}\right),\qquad T=2.
\]

For the subsequent efficient-student run, I combine the gold-label objective with Bernoulli distillation:

\[
\mathcal L = 0.7\mathcal L_{\mathrm{gold}} + 0.3\mathcal L_{\mathrm{KD}},
\]

\[
\mathcal L_{\mathrm{KD}} = \frac{T^2}{BC}
\sum_{i=1}^{B}\sum_{c=1}^{C}
\operatorname{KL}\!\left[
\operatorname{Bern}(q_{ic})\;\Vert\;
\operatorname{Bern}\!\left(\sigma(z_{S,ic}/T)\right)
\right].
\]

Independent sigmoid targets preserve the multi-label task: statutes are not forced to compete for a single probability mass. The temperature and loss mixture above are project settings.

The separate student-training notebook uses:

```python
ARCHITECTURE = "efficient"
STUDENT_MODE = "kd"
ACTION = "train"
RESUME_INPUT = None
TEACHER_CACHE_INPUT = "/kaggle/working/lsi_prefix_lorma_001_teacher_cache.zip"
```

In a new Kaggle session, attach the cache ZIP or its extracted folder and update `TEACHER_CACHE_INPUT`. The student starts from its original efficient Phase 1 initialization and checks row order, statute order, tokenizer and chunk geometry against the teacher cache. Student and KD evaluation results belong to that subsequent training run.


## References

- Bihany, H., Patel, S., and Modi, A. (2025). [LoRMA: Low-Rank Multiplicative Adaptation for LLMs](https://aclanthology.org/2025.findings-acl.527/). Findings of ACL 2025.
- Hinton, G., Vinyals, O., and Dean, J. (2015). [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531).

The embedded project source contains the implemented multi-label loss, chunk aggregation and input-identity checks. The saved run configuration records the experiment-specific adaptation, EMA and export settings.
